# Development notebook

In [5]:
import numpy as np
import pandas as pd
import xarray as xr

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# -----------------------------
# 1. Create synthetic xarray Dataset
# -----------------------------

latitudes = np.arange(45, 61, 1)      # 45, ..., 60
longitudes = np.arange(0, 11, 1)      # 0, ..., 10

# Monthly timestamps from Jan 1990 up to the current month
start = "2020-01-01"
end = pd.Timestamp.today().to_period("M").to_timestamp()
time = pd.date_range(start=start, end=end, freq="MS")

rng = np.random.default_rng(seed=42)

shape = (len(time), len(latitudes), len(longitudes))

ds = xr.Dataset(
    data_vars={
        "var1": (("time", "latitude", "longitude"), rng.normal(size=shape)),
        "var2": (("time", "latitude", "longitude"), rng.normal(size=shape)),
        "var3": (("time", "latitude", "longitude"), rng.normal(size=shape)),
        "var4": (("time", "latitude", "longitude"), rng.normal(size=shape)),
    },
    coords={
        "time": time,
        "latitude": latitudes,
        "longitude": longitudes,
    },
)

print(ds)

<xarray.Dataset> Size: 440kB
Dimensions:    (time: 78, latitude: 16, longitude: 11)
Coordinates:
  * time       (time) datetime64[ns] 624B 2020-01-01 2020-02-01 ... 2026-06-01
  * latitude   (latitude) int64 128B 45 46 47 48 49 50 51 ... 55 56 57 58 59 60
  * longitude  (longitude) int64 88B 0 1 2 3 4 5 6 7 8 9 10
Data variables:
    var1       (time, latitude, longitude) float64 110kB 0.3047 -1.04 ... 0.1543
    var2       (time, latitude, longitude) float64 110kB 1.078 ... -0.2699
    var3       (time, latitude, longitude) float64 110kB -0.2992 1.773 ... 1.625
    var4       (time, latitude, longitude) float64 110kB -1.083 ... 0.4465


In [ ]:
from dataclasses import dataclass, field
@dataclass(frozen=True)
class GriddedPCA:
    pc_scores: xr.DataArray
    pca_components: xr.DataArray
    explained_variance_ratio: xr.DataArray
    feature_mean: xr.DataArray
    feature_scale: xr.DataArray
    pca_mean: xr.DataArray
    attrs: dict

def fit_monthly_gridcell_pca(
    ds,
    variables=("var1", "var2", "var3", "var4"),
    n_components=None,
    standardize=True,
):
    """
    Fit PCA independently for each calendar month and each lat/lon grid cell.

    For each month and grid cell:
        samples  = all years for that month
        features = variables

    Returns an xarray Dataset containing everything needed to transform
    future/new data into the PCA domain.
    """

    variables = list(variables)
    n_features = len(variables)

    if n_components is None:
        n_components = n_features

    n_components = min(n_components, n_features)
    components_coord = [f"PC{i + 1}" for i in range(n_components)]

    X_all = (
        ds[variables].to_array(dim="feature").transpose("time", "latitude", "longitude", "feature")
    )

    latitudes = ds.latitude
    longitudes = ds.longitude
    months = np.arange(1, 13)

    # Historical PC scores
    pc_scores = xr.DataArray(
        np.full(
            (
                ds.sizes["time"],
                ds.sizes["latitude"],
                ds.sizes["longitude"],
                n_components,
            ),
            np.nan,
        ),
        dims=("time", "latitude", "longitude", "component"),
        coords={
            "time": ds.time,
            "latitude": latitudes,
            "longitude": longitudes,
            "component": components_coord,
        },
        name="pc_scores",
    )

    # PCA loadings / directions
    pca_components = xr.DataArray(
        np.full(
            (
                12,
                ds.sizes["latitude"],
                ds.sizes["longitude"],
                n_components,
                n_features,
            ),
            np.nan,
        ),
        dims=("month", "latitude", "longitude", "component", "feature"),
        coords={
            "month": months,
            "latitude": latitudes,
            "longitude": longitudes,
            "component": components_coord,
            "feature": variables,
        },
        name="pca_components",
    )

    # Explained variance ratio
    explained_variance_ratio = xr.DataArray(
        np.full(
            (
                12,
                ds.sizes["latitude"],
                ds.sizes["longitude"],
                n_components,
            ),
            np.nan,
        ),
        dims=("month", "latitude", "longitude", "component"),
        coords={
            "month": months,
            "latitude": latitudes,
            "longitude": longitudes,
            "component": components_coord,
        },
        name="explained_variance_ratio",
    )

    # Means and scales used before PCA
    feature_mean = xr.DataArray(
        np.full(
            (
                12,
                ds.sizes["latitude"],
                ds.sizes["longitude"],
                n_features,
            ),
            np.nan,
        ),
        dims=("month", "latitude", "longitude", "feature"),
        coords={
            "month": months,
            "latitude": latitudes,
            "longitude": longitudes,
            "feature": variables,
        },
        name="feature_mean",
    )

    feature_scale = xr.DataArray(
        np.full(
            (
                12,
                ds.sizes["latitude"],
                ds.sizes["longitude"],
                n_features,
            ),
            np.nan,
        ),
        dims=("month", "latitude", "longitude", "feature"),
        coords={
            "month": months,
            "latitude": latitudes,
            "longitude": longitudes,
            "feature": variables,
        },
        name="feature_scale",
    )

    # Mean internally subtracted by PCA after scaling/centering
    pca_mean = xr.DataArray(
        np.full(
            (
                12,
                ds.sizes["latitude"],
                ds.sizes["longitude"],
                n_features,
            ),
            np.nan,
        ),
        dims=("month", "latitude", "longitude", "feature"),
        coords={
            "month": months,
            "latitude": latitudes,
            "longitude": longitudes,
            "feature": variables,
        },
        name="pca_mean",
    )

    for month in months:
        print("Month: " + str(month))
        month_mask = ds.time.dt.month == month
        month_times = ds.time.where(month_mask, drop=True)

        for lat in latitudes.values:
            for lon in longitudes.values:
                X = X_all.sel(time=month_times, latitude=lat, longitude=lon).values

                valid_rows = np.isfinite(X).all(axis=1)
                X_valid = X[valid_rows]

                if X_valid.shape[0] < 2:
                    continue

                ncomp_here = min(n_components, X_valid.shape[0], n_features)

                if standardize:
                    scaler = StandardScaler()
                    X_prepared = scaler.fit_transform(X_valid)

                    mean_here = scaler.mean_
                    scale_here = scaler.scale_

                else:
                    mean_here = X_valid.mean(axis=0)
                    scale_here = np.ones(n_features)

                    X_prepared = X_valid - mean_here

                pca = PCA(n_components=ncomp_here)
                scores = pca.fit_transform(X_prepared)

                valid_times = month_times.values[valid_rows]

                pc_scores.loc[
                    dict(
                        time=valid_times,
                        latitude=lat,
                        longitude=lon,
                        component=components_coord[:ncomp_here],
                    )
                ] = scores

                pca_components.loc[
                    dict(
                        month=month,
                        latitude=lat,
                        longitude=lon,
                        component=components_coord[:ncomp_here],
                        feature=variables,
                    )
                ] = pca.components_

                explained_variance_ratio.loc[
                    dict(
                        month=month,
                        latitude=lat,
                        longitude=lon,
                        component=components_coord[:ncomp_here],
                    )
                ] = pca.explained_variance_ratio_

                feature_mean.loc[
                    dict(month=month, latitude=lat, longitude=lon, feature=variables)
                ] = mean_here

                feature_scale.loc[
                    dict(month=month, latitude=lat, longitude=lon, feature=variables)
                ] = scale_here

                pca_mean.loc[dict(month=month, latitude=lat, longitude=lon, feature=variables)] = (
                    pca.mean_
                )

    return GriddedPCA(
        pc_scores= pc_scores,
            pca_components= pca_components,
            explained_variance_ratio= explained_variance_ratio,
            feature_mean= feature_mean,
            feature_scale= feature_scale,
            pca_mean= pca_mean,
            attrs={
            "variables": ",".join(variables),
            "standardize": standardize,
        },
    )


In [9]:
pca_ds = fit_monthly_gridcell_pca(ds, n_components=2)

print(pca_ds)


Month: 1
Latitude : 45
Latitude : 46
Latitude : 47
Latitude : 48
Latitude : 49
Latitude : 50
Latitude : 51
Latitude : 52
Latitude : 53
Latitude : 54
Latitude : 55
Latitude : 56
Latitude : 57
Latitude : 58
Latitude : 59
Latitude : 60
Month: 2
Latitude : 45
Latitude : 46
Latitude : 47
Latitude : 48
Latitude : 49
Latitude : 50
Latitude : 51
Latitude : 52
Latitude : 53
Latitude : 54
Latitude : 55
Latitude : 56
Latitude : 57
Latitude : 58
Latitude : 59
Latitude : 60
Month: 3
Latitude : 45
Latitude : 46
Latitude : 47
Latitude : 48
Latitude : 49
Latitude : 50
Latitude : 51
Latitude : 52
Latitude : 53
Latitude : 54
Latitude : 55
Latitude : 56
Latitude : 57
Latitude : 58
Latitude : 59
Latitude : 60
Month: 4
Latitude : 45
Latitude : 46
Latitude : 47
Latitude : 48
Latitude : 49
Latitude : 50
Latitude : 51
Latitude : 52
Latitude : 53
Latitude : 54
Latitude : 55
Latitude : 56
Latitude : 57
Latitude : 58
Latitude : 59
Latitude : 60
Month: 5
Latitude : 45
Latitude : 46
Latitude : 47
Latitude : 48
Lat

In [10]:
pca_ds.sel(month=2, latitude=50, longitude=5)


AttributeError: 'GriddedPCA' object has no attribute 'sel'

In [8]:
pc1_feb = (
    pca_ds["pc_scores"]
    .sel(latitude=50, longitude=5, component="PC1")
    .where(ds.time.dt.month == 2, drop=True)
)

print(pc1_feb)


<xarray.DataArray 'pc_scores' (time: 37)> Size: 296B
array([ 1.32072285,  0.13444803, -1.34012768, -0.6648799 , -0.24592705,
       -0.57162839,  1.5515406 ,  0.91552125, -0.22149426, -1.72609809,
       -1.01704143,  0.07211459,  1.19907287,  0.42352033,  0.96081722,
       -0.13564307, -0.15277324,  2.77245556, -3.53578946, -0.10491216,
        0.76302134, -0.0411069 ,  0.12892743, -0.43547721,  1.43544465,
        1.50687306, -0.74461149, -1.42089571,  0.39468312,  0.51951524,
        1.38195065,  0.89625063, -0.16049863, -0.1529431 , -1.75963928,
       -0.26964842, -1.67574397])
Coordinates:
  * time       (time) datetime64[ns] 296B 1990-02-01 1991-02-01 ... 2026-02-01
    latitude   int64 8B 50
    longitude  int64 8B 5
    component  <U3 12B 'PC1'


In [9]:
pca_ds["explained_variance_ratio"].sel(
    month=2,
    latitude=50,
    longitude=5,
)


<xarray.DataArray 'explained_variance_ratio' (component: 2)> Size: 16B
array([0.34690717, 0.26076142])
Coordinates:
    latitude   int64 8B 50
    longitude  int64 8B 5
  * component  (component) <U3 24B 'PC1' 'PC2'
    month      int64 8B 2

In [14]:
def transform_new_grid_to_pca(
    new_ds,
    pca_model : GriddedPCA,
    variables : tuple["str"] =("var1", "var2", "var3", "var4"),
):
    """
    Transform new data into the pre-fitted monthly/grid-cell PCA domain.

    new_ds must have:
        dims: time, latitude, longitude
        variables: var1, var2, var3, var4

    Uses the PCA model fitted for:
        month(new time) × latitude × longitude
    """

    variables = list(variables)

    X_new_all = (
        new_ds[variables]
        .to_array(dim="feature")
        .transpose("time", "latitude", "longitude", "feature")
    )

#    components_coord = pca_model.component.values
    components_coord = pca_model.pca_components.component.values


    new_pc_scores = xr.DataArray(
        np.full(
            (
                new_ds.sizes["time"],
                new_ds.sizes["latitude"],
                new_ds.sizes["longitude"],
                len(components_coord),
            ),
            np.nan,
        ),
        dims=("time", "latitude", "longitude", "component"),
        coords={
            "time": new_ds.time,
            "latitude": new_ds.latitude,
            "longitude": new_ds.longitude,
            "component": components_coord,
        },
        name="pc_scores",
    )

    for t in new_ds.time.values:
        month = pd.Timestamp(t).month

        for lat in new_ds.latitude.values:
            for lon in new_ds.longitude.values:
                x = X_new_all.sel(time=t, latitude=lat, longitude=lon).values

                if not np.isfinite(x).all():
                    continue

                mean = (
                    pca_model.feature_mean.sel(month=month, latitude=lat, longitude=lon).values
                )

                scale = (
                    pca_model.feature_scale.sel(month=month, latitude=lat, longitude=lon).values
                )

                pca_mean = (
                    pca_model.pca_mean.sel(month=month, latitude=lat, longitude=lon).values
                )

                components = (
                    pca_model.pca_components.sel(month=month, latitude=lat, longitude=lon).values
                )

                valid_components = np.isfinite(components).all(axis=1)

                if not valid_components.any():
                    continue

                components_valid = components[valid_components]

                # Same transformation as sklearn:
                # 1. Standardize/center using training parameters
                # 2. Subtract PCA mean
                # 3. Project onto PCA components
                x_prepared = (x - mean) / scale
                x_centered_for_pca = x_prepared - pca_mean

                scores = x_centered_for_pca @ components_valid.T

                new_pc_scores.loc[
                    dict(
                        time=t,
                        latitude=lat,
                        longitude=lon,
                        component=components_coord[valid_components],
                    )
                ] = scores

    return new_pc_scores


In [15]:
next_time = pd.Timestamp(ds.time.values[-1]) + pd.DateOffset(months=1)

new_shape = (
    1,
    ds.sizes["latitude"],
    ds.sizes["longitude"],
)

rng = np.random.default_rng(seed=123)

new_ds = xr.Dataset(
    data_vars={
        "var1": (("time", "latitude", "longitude"), rng.normal(size=new_shape)),
        "var2": (("time", "latitude", "longitude"), rng.normal(size=new_shape)),
        "var3": (("time", "latitude", "longitude"), rng.normal(size=new_shape)),
        "var4": (("time", "latitude", "longitude"), rng.normal(size=new_shape)),
    },
    coords={
        "time": [next_time],
        "latitude": ds.latitude,
        "longitude": ds.longitude,
    },
)


In [16]:
new_pc_scores = transform_new_grid_to_pca(
    new_ds,
    pca_ds,
    variables=("var1", "var2", "var3", "var4"),
)

print(new_pc_scores)


<xarray.DataArray 'pc_scores' (time: 1, latitude: 16, longitude: 11,
                               component: 2)> Size: 3kB
array([[[[-2.15291915, -2.69500077],
         [-0.01658273,  1.51041667],
         [ 1.07817162,  0.80895646],
         [ 0.64313086,  0.19097287],
         [-0.84625124, -1.61994165],
         [-1.33730029, -3.16839869],
         [-0.67082235,  0.23107415],
         [-0.64643076,  0.04498025],
         [-0.68845668,  1.32786733],
         [ 0.80611941,  0.08640492],
         [ 0.65998914,  1.31575099]],

        [[-0.04185899, -1.03299555],
         [-1.19858858, -0.41333225],
         [ 0.4899024 ,  1.5987087 ],
         [-0.55789633,  2.38784587],
         [ 0.78841884, -0.07448611],
         [ 1.06798814,  0.55596728],
         [-0.44259094, -0.22236128],
         [-1.55813768, -2.63621276],
...
         [ 1.80718354,  1.00947955],
         [ 1.3819913 ,  0.35679195],
         [-1.32277046, -2.14244095],
         [-1.83524134, -0.8250435 ],
         [ 1.32368

In [71]:
from scipy.spatial.distance import mahalanobis


def mahalanobis_candidates_month(PCA_model : GriddedPCA, New_PCA_scores : xr.DataArray) -> tuple[xr.DataArray]:
    """Compute Mahalanobis distance for each candidate, for each grid cell."""
    month = New_PCA_scores.time.dt.month.values
    if len(month) > 1:
        raise ValueError("Multiple months are not directly handled in this function!")
    month_mask = PCA_model.pc_scores.time.dt.month == month
    month_times = PCA_model.pc_scores.time.where(month_mask, drop=True)
    PCA_candidates = PCA_model.pc_scores.sel(time=month_times)
    mahalanobis_spatial = xr.DataArray(
            np.full(
                (
                    New_PCA_scores.sizes["latitude"],
                    New_PCA_scores.sizes["longitude"],
                    PCA_candidates.sizes["time"],
                ),
                np.nan,
            ),
            dims=( "latitude", "longitude", "candidate"),
            coords={
                "latitude": New_PCA_scores.latitude,
                "longitude": New_PCA_scores.longitude,
                "candidate": PCA_candidates.time.values,
            },
            attrs={
                  "time": New_PCA_scores.time,
            },
            name="mahalanobis",
        )
    
    mahalanobis_score = xr.DataArray(
        np.full(
            (
                New_PCA_scores.sizes["latitude"],
                New_PCA_scores.sizes["longitude"],
                PCA_candidates.sizes["time"],
            ),
            np.nan,
        ),
        dims=("latitude", "longitude", "candidate"),
        coords={
            "latitude": New_PCA_scores.latitude,
            "longitude": New_PCA_scores.longitude,
            "candidate": PCA_candidates.time.values,
        },
        attrs={
            "time": New_PCA_scores.time,
        },
        name="mahalanobis_score",
    )

    i = 0
    new_pca = New_PCA_scores.isel(time=0)
    variance_matrix = PCA_model.explained_variance_ratio.sel(month=month)
    for lat in New_PCA_scores.latitude:
        for lon in New_PCA_scores.longitude:
            new_pca_latlon = new_pca.sel(latitude=lat, longitude=lon)
            for candidate in PCA_candidates.time:
                dist_new = mahalanobis(
                    u=PCA_candidates.sel(latitude=lat, longitude=lon, time=candidate),
                    v=new_pca_latlon,
                    VI=np.diag(variance_matrix.sel(latitude=lat, longitude=lon).values[0]),
                )

                mahalanobis_spatial.loc[
                    dict(
                        latitude=lat,
                        longitude=lon,
                        candidate=candidate,
                    )
                ] = dist_new

                mahalanobis_score.loc[
                    dict(
                        latitude=lat,
                        longitude=lon,
                        candidate=candidate,
                    )
                ] = 1/dist_new
                
    return mahalanobis_spatial, mahalanobis_score

In [72]:
distance_candidates, score_candidates = mahalanobis_candidates_month(PCA_model=pca_ds,
                             New_PCA_scores=new_pc_scores)

In [80]:
def select_candidate(mahalanobis_score : xr.DataArray):
    # 1 - Compute score for each candidate:
    candidate_scores = (
        mahalanobis_score.mean(["latitude", "longitude"])
        .to_dataframe()
        .sort_values(by="mahalanobis_score",ascending=False)
    )
    print(candidate_scores)
    # 2 - compare and select the best
    return candidate_scores.index[0]

In [81]:
select_candidate(score_candidates)

            mahalanobis_score
candidate                    
2021-07-01           1.166528
2022-07-01           1.074383
2020-07-01           1.072601
2023-07-01           1.041864
2025-07-01           0.977840
2024-07-01           0.899105


Timestamp('2021-07-01 00:00:00')

In [1]:
import pandas as pd


tst = pd.Timestamp("2021-07-01 00:00:00")

In [3]:
diagnostics = {
    "best_year": 1,
    "best_month": 10,
    "best_score": 3,
}


In [4]:
diagnostics["best_month"]

10